In [ ]:
#Basic from scratch decision tree classifier implementation in Python


#Dependencies
import math
import numpy as np
import pandas as pd


In [1]:
#First we define the node class

class Node():

    def __init__(self,threshhold: float = None, Leaf: bool = False, ds_y: list = [], index:int = 0, ds_X: list= []):

        self.threshold = threshhold
        self.Leaf = Leaf
        self.ds_y = ds_y
        self.ds_X = ds_X
        self.index = index
        self.left = None
        self.right = None

        if Leaf == True:
            self.leaf_node()

    def evaluate(self, value):
        feature_val = value[self.index]

        if self.Leaf == True:
            self.leaf_node()
            base = 0
            for key in self.class_dict:
                if self.class_dict[key] > base:
                    base = self.class_dict[key]
                    name = key
            return self.class_dict
        
        if feature_val < self.threshold:
            return self.left.evaluate(value)
        else:
            return self.right.evaluate(value)
        
    def leaf_node(self):
        
        ds_y = self.ds_y
        class_dict: dict = {}
        total_instances: int = len(ds_y)

        for unique_class in set(ds_y):
            class_dict[unique_class] = 0
        
        for class_instance in ds_y:
            class_dict[class_instance] +=1
        
        for unique_class in class_dict:
            class_dict[unique_class] = class_dict[unique_class] / total_instances

        self.class_dict = class_dict

class DecisionTree():

    def __init__(self):
        pass

    def __str__(self):
        return 'You have not trained a model yet'

    def gini(self, ds_y: list) -> float:
        '''
        Gini = 1 - Sum((proportion of set)**2)
        '''
        gini:float = 0.0
        gini_dict: dict = {}
        no_elements: float = float(len(ds_y))
        for value in ds_y:
            if value not in gini_dict:
                gini_dict[value] = 0  
            gini_dict[value] += 1.0
        
        for key in gini_dict:
            gini_dict[key] = (gini_dict[key]/no_elements)**2
            gini += gini_dict[key]

        return 1-gini

    def weighted_gini(self, ds_y_L, ds_y_R):

        gini_L = self.gini(ds_y_L)
        gini_R = self.gini(ds_y_R)

        proportion_left = len(ds_y_L)
        proportion_right = len(ds_y_R)
        total_proportion = proportion_left+proportion_right
        
        weighted_gini =(((gini_L)*(proportion_left/total_proportion))+((gini_R)*((proportion_right/total_proportion))))
        
        return weighted_gini

    def split_data(self, threshold, feature_index, ds_X, ds_y):
        ds_X_R = []
        ds_y_R = []

        ds_X_L = []
        ds_y_L = []

        for i in range(len(ds_X)):
            if ds_X[i][feature_index] > threshold:
                ds_X_R.append(ds_X[i])
                ds_y_R.append(ds_y[i])

            else:
                ds_X_L.append(ds_X[i])
                ds_y_L.append(ds_y[i])

        return ds_X_L, ds_y_L, ds_X_R, ds_y_R

    def split(self, ds_X, ds_y):
        best_gini: list = [1000000]
        for features in ds_X:
                for feature_index, threshhold in enumerate(features):
                    ds_X_L, ds_y_L, ds_X_R, ds_y_R = self.split_data(threshhold, feature_index, ds_X, ds_y)

                    current_gini = self.weighted_gini(ds_y_L, ds_y_R)
                    current_ws = best_gini[0]

                    if current_gini<current_ws:
                        best_gini = [current_gini, threshhold, feature_index, ds_X_L, ds_X_R, ds_y_L, ds_y_R]

        return None if best_gini[0] == 1000000 else best_gini

    def fit(self, ds_X = list(), ds_y = list(), model = None, max_depth = 3, min_samples_split = 10, min_samples_leaf = 10, min_impurity_split = 0.1, previous_node = []):
        #[weighted_sum, threshhold, threshhold_index, ds_X_L, ds_X_R, ds_Y_L, ds_Y_R]

        if isinstance(ds_X, list) == False:
            ds_X.values().tolist()

        if isinstance(ds_y, list) == False:
            ds_y.values().tolist()

        if model == None:
            model = Node(ds_y = ds_y , ds_X = ds_X)
            self.model = model

        if len(ds_y) > min_samples_leaf and isinstance(previous_node, list) is False:
            previous_node.leaf_node()
            return
        
        max_depth -= 1
        
        if len(model.ds_y) < min_samples_split:
            model.Leaf = True
            return 

        if self.gini(model.ds_y) < min_impurity_split:
            model.Leaf = True
            return 

        if max_depth > 0: 
            split = self.split(model.ds_X, model.ds_y)

            if split == None:
                model.Leaf = True
                return

            model.threshold = split[1]

            model.index = split[2]
            if max_depth == 1:
            
                model.left = Node(ds_y = split[5], ds_X = split[3], Leaf = True)

                model.right = Node(ds_y = split[6], ds_X = split[4], Leaf = True)

                model.left.leaf_node()
                model.right.leaf_node()
                return
            else:
                model.left = Node(ds_y = split[5], ds_X = split[3])

                model.right = Node(ds_y = split[6], ds_X = split[4])
            
            self.fit(model = model.left, max_depth=max_depth, min_impurity_split = min_impurity_split, min_samples_split = min_samples_split, previous_node = model, min_samples_leaf = min_samples_leaf)
            #L and R
            self.fit(model = model.right, max_depth=max_depth, min_impurity_split = min_impurity_split, min_samples_split = min_samples_split, previous_node = model, min_samples_leaf = min_samples_leaf)

    def predict(self, df_X):
        predictions: list = []
        for feature_value in df_X:

            if not isinstance(feature_value, list):
                return self.model.evaluate(df_X)
            predictions.append(self.model.evaluate(feature_value))
        
        return predictions
    
    def train_test_split():
        pass


    def accuracy_score(df_y, predictions):
        pass
    '''
    To do:

    numpy implementation for effeciency

    Unit tests for decision tree + Lin reg (see if it produces the same results as sklearn)
    '''

